This notebook holds the code for the integration of the multiple samples, using protein and RNA data in an adata object. This object was transferred from a Seurat object using AnndataR.
The Seurat object had 2 assays; RNA and protein. Next, 2 AnnData objects were joined to make a mudata object, which holds the RNA and protein assays.

Load necessary packages

In [ ]:
import scanpy as sc
import mudata
import muon 
import scvi
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
scvi.settings.seed = 0
print("Last run with scvi-tools version:", scvi.__version__)
scvi.settings.progress_bar_style = "tqdm"

Load data

In [ ]:
mdata = mudata.read_h5mu("c:/Users/irc/Desktop/Internship Bioinformatics 2025-2026/Lode/Integration_totalVI/mdata.h5mu")

Standard preprocessing

In [ ]:
# get HVG, since all genes would require way longer training, more memory and more noise
sc.pp.highly_variable_genes(
    mdata.mod["RNA"],
    n_top_genes=4000,
    batch_key="orig.ident",  # or your chosen batch key, which is the information from the different experiments
    flavor="seurat_v3", # can be confusing that the data was not scaled or normalized before this step, but Seurat v3 expects raw counts
    layer=None # None indicates the default layer
)

mdata.mod["rna_subset"] = mdata.mod["RNA"][:, mdata.mod["RNA"].var["highly_variable"]].copy()

In [ ]:
mdata.update()

In [ ]:
mdata

In [ ]:
# remove typo
mdata.mod["RNA"].obs["experiment"] = mdata.mod["RNA"].obs["experiment"].replace(
    "CITEseq_LNP_pIC_LNPS",
    "CITEseq_LNP_pIC_LNPs"
)
mdata.mod["protein"].obs["experiment"] = mdata.mod["protein"].obs["experiment"].replace(
    "CITEseq_LNP_pIC_LNPS",
    "CITEseq_LNP_pIC_LNPs"
)
mdata.mod["rna_subset"].obs["experiment"] = mdata.mod["rna_subset"].obs["experiment"].replace(
    "CITEseq_LNP_pIC_LNPS",
    "CITEseq_LNP_pIC_LNPs"
)

mdata.update()

In [ ]:
mdata.mod["protein"].obs["orig.ident"] # batch layer 

3 proteins have zero expression across all batches, which will be removed now, to reduce computational load and noise

In [ ]:
# 1. Identify non-zero proteins (sum across all cells > 0)
protein_counts = mdata['protein'].X
protein_sums = np.asarray(protein_counts.sum(axis=0)).flatten()
expressed_proteins = mdata['protein'].var_names[protein_sums > 0]

print(f"Removing {len(mdata['protein'].var_names) - len(expressed_proteins)} zero-expression proteins.")

# 2. Subset the protein modality in place
mdata.mod['protein'] = mdata['protein'][:, expressed_proteins].copy()
mdata.update()

In [ ]:
mdata

In [ ]:
# prepare model
scvi.model.TOTALVI.setup_mudata(
    mdata,
    rna_layer=None, # X layer
    protein_layer=None, #X layer
    batch_key="orig.ident",
    modalities={
        "rna_layer": "rna_subset",
        "protein_layer": "protein",
        "batch_key": "rna_subset",
    },
)

The command below returns that some proteins are all zero in some batches, which can be biologically true, but they can also not have been used in those batches

In [ ]:

# this snippet returns a table of proteins that are 0 in at least one batch, which can be useful for filtering or understanding the data
# True means a zero count in that batch, False means non-zero count

# Extract protein matrix and batch labels
protein_mat = mdata['protein'].X
protein_df = pd.DataFrame(
    protein_mat.toarray() if hasattr(protein_mat, "toarray") else protein_mat,
    columns=mdata['protein'].var_names
)
protein_df['batch'] = mdata['rna_subset'].obs['orig.ident'].values

# Calculate total counts per protein in each batch
batch_sums = protein_df.groupby('batch').sum()

# Find which batch-protein pairs are exactly 0
zero_batch_protein = (batch_sums == 0)

# Print any protein that is 0 in at least one batch
missing_in_batches = zero_batch_protein.loc[:, zero_batch_protein.any()]
print("Proteins with 0 counts in specific batches:")
print(missing_in_batches)

In [ ]:
# run model
model = scvi.model.TOTALVI(mdata, 
                           override_missing_proteins=False,
                           empirical_protein_background_prior=False) # without this, totalVI tries to fit a Gaussian Mixture Model on 100 cells per batch to initialize
                                                                     # background priors before training, which will hit a shape mismatch (1,220) versus (220, )
                                                                     # with this parameter false, it initializes those background priors randomly, and learns them during training. 
                                                                     # This is a better approach for small datasets, but may be less stable for larger datasets. 

# the ovveride_missing_proteins=True tells the model that the zero counts are biologically meaningful and not due to the antibody not being present in the panel
# setting this to False will treat the zero counts as missing values

In [ ]:
# train model, with max epochs 
model.train(max_epochs = 1000,
            early_stopping=True,
            accelerator = "cpu",
            plan_kwargs = {"lr": 1e-3},
            check_val_every_n_epoch = 10,
            enable_progress_bar = True
            )

In [ ]:
# save model
model.save("totalVI_integration_model", overwrite=True)

In [ ]:
fig, ax = plt.subplots(1, 1)
model.history["elbo_train"].plot(ax=ax, label="train")
model.history["elbo_validation"].plot(ax=ax, label="validation")
ax.set(title="Negative ELBO over training epochs", ylim=(1200, 1400))
ax.legend()

Analyze output of the model. Also later compare both integrations on the same UMAP coordinates

In [ ]:
# Load the saved model folder
model = scvi.model.TOTALVI.load(
    dir_path="c:/Users/irc/Desktop/Internship Bioinformatics 2025-2026/Lode/Integration_totalVI/totalVI_integration_model",
    adata=mdata,            # Or your main AnnData/MuData object
    accelerator="cpu"       # Forces CPU execution for stability
)

print("✓ TOTALVI model loaded successfully!")

In [ ]:
rna = mdata.mod["rna_subset"]
protein = mdata.mod["protein"]

In [ ]:
# store the latent representation in the adata object
TOTALVI_LATENT_KEY = "X_totalVI"
rna.obsm[TOTALVI_LATENT_KEY] = model.get_latent_representation()

In [ ]:
# this block extracts the denoised RNA and protein expression values, as wel as the protein foreground probabilities, and stores them in their respective layers
rna_denoised, protein_denoised = model.get_normalized_expression(
    n_samples=1, return_mean=True, transform_batch=['JVE008', 'JVE010', 'SAM016', 'SAM05', 'SAM06', 'SAM2', 'SAM3',
 'VBO004', 'VBO005', 'VBO006', 'VBO007', 'VBO008', 'VBO009', 'VBO010',
 'VBO011', 'VBO012']
)
rna.layers["denoised_rna"] = rna_denoised
protein.layers["denoised_protein"] = protein_denoised

protein.layers["protein_foreground_prob"] = 100 * model.get_protein_foreground_probability(
    n_samples=1, return_mean=True, transform_batch=['JVE008', 'JVE010', 'SAM016', 'SAM05', 'SAM06', 'SAM2', 'SAM3',
 'VBO004', 'VBO005', 'VBO006', 'VBO007', 'VBO008', 'VBO009', 'VBO010',
 'VBO011', 'VBO012']
)

mdata.update()

Compute clusters and visualize latent space

In [ ]:
TOTALVI_CLUSTERS_KEY = "leiden_totalVI"

sc.pp.neighbors(rna, use_rep=TOTALVI_LATENT_KEY)
sc.tl.umap(rna)
sc.tl.leiden(rna, key_added=TOTALVI_CLUSTERS_KEY)

In [ ]:
mdata.update()

We can now use muon plotting functions which can pull data from either modality of the MuData object.

In [ ]:
mdata

In [ ]:
# we did totalVI on the multimodal data
# before this block, the UMAP coordinates were calculated based on a neighbourhood graph which was built on the totalVI latent space
# these UMAP coordinates were saved in the rna_subset modality under .obsm["X_umap"], so we need to call that modality for the information
muon.pl.embedding(
    mdata,
    basis="rna_subset:X_umap",
    color=[f"rna_subset:{TOTALVI_CLUSTERS_KEY}", "rna_subset:orig.ident", "rna_subset:celltype_new", "rna_subset:experiment", "rna_subset:treatment"],
    frameon=False,
    ncols=2
)

Write out object (integrated)

In [ ]:
# some columns are problematic and prevent writing, so we will just remove these as they are not essential
# Drop all 'vf_vst_counts' columns from mdata.var and modalities
vst_cols = [c for c in mdata.var.columns if c.startswith("vf_vst_counts")]
mdata.var.drop(columns=vst_cols, inplace=True, errors="ignore")

for mod in mdata.mod.keys():
    mod_vst_cols = [c for c in mdata.mod[mod].var.columns if c.startswith("vf_vst_counts")]
    mdata.mod[mod].var.drop(columns=mod_vst_cols, inplace=True, errors="ignore")

mdata.update()

In [ ]:
mdata.write("Integrated_totalVI.h5mu")